# AWS Data Engineering — First Contact

AWS data engineering is easiest to understand as a layered stack: **S3 stores data**, **Glue catalogs metadata**, **Athena queries files with SQL**, **EMR runs Spark at scale**, and **Redshift serves warehouse-style analytics**. Storage, metadata, and compute are intentionally separated so teams can scale each independently instead of tying everything to one always-on database.

A second mental model is **serverless vs managed**. Serverless services such as S3, Glue Catalog, and Athena let you avoid cluster management and pay mainly for what you use. Managed services such as EMR and Redshift give you more tuning power and predictable performance, but you are also responsible for more operational choices around sizing, runtime, and cost.

Citi's cloud DE stack: raw telemetry lands in S3, Glue catalogs the schema, Athena runs ad-hoc SQL, EMR runs batch Spark jobs — no servers to manage.

```text
[Postgres] → [Export CSV] → [S3 Data Lake] → [Glue Catalog] → [Athena SQL] → [Results]
```

In [1]:
import os
# Use the study AWS profile (account 357811130281, us-east-1)
os.environ["AWS_PROFILE"] = "study"
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"

%pip install boto3 psycopg2-binary pandas pyarrow

import boto3
import psycopg2
import pandas as pd
import io
import json
import os
import time
from botocore.exceptions import ClientError

account_id = boto3.client('sts').get_caller_identity()['Account']
region = boto3.session.Session().region_name
print(f"AWS Account: {account_id}, Region: {region}")

BUCKET_NAME = f"citi-telemetry-lake-{account_id}"
ATHENA_BUCKET = f"citi-telemetry-athena-{account_id}"
GLUE_DB = "citi_telemetry_db"

Note: you may need to restart the kernel to use updated packages.


AWS Account: 357811130281, Region: us-east-1


Create the data lake and Athena results buckets

In [2]:
s3 = boto3.client('s3')

def create_bucket_safe(bucket_name: str):
    try:
        if region == 'us-east-1':
            s3.create_bucket(Bucket=bucket_name)
        else:
            s3.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={'LocationConstraint': region}
            )
        print(f"Bucket created: {bucket_name}")
    except ClientError as e:
        code = e.response.get('Error', {}).get('Code', '')
        if code in ('BucketAlreadyOwnedByYou', 'BucketAlreadyExists'):
            print(f"Bucket already exists and is usable: {bucket_name}")
        else:
            print(f"Error creating bucket {bucket_name}: {e}")

def block_public_access(bucket_name: str):
    try:
        s3.put_public_access_block(
            Bucket=bucket_name,
            PublicAccessBlockConfiguration={
                'BlockPublicAcls': True,
                'IgnorePublicAcls': True,
                'BlockPublicPolicy': True,
                'RestrictPublicBuckets': True,
            }
        )
        print(f"Public access blocked: {bucket_name}")
    except ClientError as e:
        print(f"Error blocking public access on {bucket_name}: {e}")

def enable_versioning(bucket_name: str):
    try:
        s3.put_bucket_versioning(
            Bucket=bucket_name,
            VersioningConfiguration={'Status': 'Enabled'}
        )
        print(f"Versioning enabled: {bucket_name}")
    except ClientError as e:
        print(f"Error enabling versioning on {bucket_name}: {e}")

create_bucket_safe(BUCKET_NAME)
create_bucket_safe(ATHENA_BUCKET)
enable_versioning(BUCKET_NAME)
block_public_access(BUCKET_NAME)
block_public_access(ATHENA_BUCKET)

print(f"Created: {BUCKET_NAME} and {ATHENA_BUCKET}")

Bucket created: citi-telemetry-lake-357811130281


Bucket created: citi-telemetry-athena-357811130281
Versioning enabled: citi-telemetry-lake-357811130281


Public access blocked: citi-telemetry-lake-357811130281
Public access blocked: citi-telemetry-athena-357811130281
Created: citi-telemetry-lake-357811130281 and citi-telemetry-athena-357811130281


## Export Telemetry Data to S3

Parquet is the standard DE format for S3 data lakes because it is columnar, compressed, and embeds schema information. That makes it cheaper to scan in Athena and faster for analytics-style workloads than plain CSV.

In [3]:
pg_conn = None

try:
    pg_conn = psycopg2.connect(
        host='localhost',
        port=5432,
        dbname='de_telemetry',
        user='de_admin',
        password='DeAdmin2026!'
    )
    print('Connected to Postgres')
except Exception as e:
    print(f'Error connecting to Postgres: {e}')
    raise

endpoints_df = pd.read_sql('SELECT endpoint_id, name, region, status, category FROM endpoints', pg_conn)
alerts_df = pd.read_sql('SELECT alert_id, endpoint_id, severity, message, created_at FROM alerts', pg_conn)

endpoints_df['endpoint_id'] = endpoints_df['endpoint_id'].astype('int32')
alerts_df['alert_id'] = alerts_df['alert_id'].astype('int32')
alerts_df['endpoint_id'] = alerts_df['endpoint_id'].astype('int32')
alerts_df['created_at'] = alerts_df['created_at'].astype(str)

def upload_dataframe_as_parquet(df: pd.DataFrame, bucket: str, key: str):
    try:
        buffer = io.BytesIO()
        df.to_parquet(buffer, index=False)
        buffer.seek(0)
        s3.put_object(Bucket=bucket, Key=key, Body=buffer.getvalue())
        print(f'Uploaded s3://{bucket}/{key}')
    except ClientError as e:
        print(f'Error uploading {key}: {e}')
        raise

upload_dataframe_as_parquet(endpoints_df, BUCKET_NAME, 'telemetry/endpoints/endpoints.parquet')
upload_dataframe_as_parquet(alerts_df, BUCKET_NAME, 'telemetry/alerts/alerts.parquet')

print(f"Uploaded endpoints ({len(endpoints_df)} rows) and alerts ({len(alerts_df)} rows) to S3")

Connected to Postgres

C:\Users\shareuser\AppData\Local\Temp\ipykernel_61932\3700034252.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  endpoints_df = pd.read_sql('SELECT endpoint_id, name, region, status, category FROM endpoints', pg_conn)
C:\Users\shareuser\AppData\Local\Temp\ipykernel_61932\3700034252.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  alerts_df = pd.read_sql('SELECT alert_id, endpoint_id, severity, message, created_at FROM alerts', pg_conn)


Uploaded s3://citi-telemetry-lake-357811130281/telemetry/endpoints/endpoints.parquet


Uploaded s3://citi-telemetry-lake-357811130281/telemetry/alerts/alerts.parquet
Uploaded endpoints (10000 rows) and alerts (25000 rows) to S3


## AWS Glue — The Metadata Catalog

Glue Data Catalog is the metadata layer for the lake. It stores databases, tables, columns, and locations in S3. Athena, EMR, and Redshift Spectrum can all read from it. In practice, it plays the role of a Hive metastore in AWS. Glue crawlers can auto-discover schema, but in this notebook we create the catalog objects directly with boto3 so the learning path is explicit.

In [4]:
glue = boto3.client('glue')

try:
    glue.create_database(
        DatabaseInput={
            'Name': GLUE_DB,
            'Description': 'Citi telemetry catalog for S3-backed lake data'
        }
    )
    print(f'Glue database created: {GLUE_DB}')
except glue.exceptions.AlreadyExistsException:
    print(f'Glue database already exists: {GLUE_DB}')
except ClientError as e:
    print(f'Error creating Glue database {GLUE_DB}: {e}')

def create_glue_table(table_name: str, s3_location: str, columns: list):
    try:
        glue.create_table(
            DatabaseName=GLUE_DB,
            TableInput={
                'Name': table_name,
                'TableType': 'EXTERNAL_TABLE',
                'Parameters': {
                    'classification': 'parquet',
                    'EXTERNAL': 'TRUE'
                },
                'StorageDescriptor': {
                    'Columns': columns,
                    'Location': s3_location,
                    'InputFormat': 'org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat',
                    'OutputFormat': 'org.apache.hadoop.hive.ql.io.parquet.MapredParquetOutputFormat',
                    'Compressed': False,
                    'SerdeInfo': {
                        'SerializationLibrary': 'org.apache.hadoop.hive.ql.io.parquet.serde.ParquetHiveSerDe',
                        'Parameters': {'serialization.format': '1'}
                    }
                }
            }
        )
        print(f'Glue table created: {GLUE_DB}.{table_name}')
    except glue.exceptions.AlreadyExistsException:
        print(f'Glue table already exists: {GLUE_DB}.{table_name}')
    except ClientError as e:
        print(f'Error creating table {GLUE_DB}.{table_name}: {e}')

create_glue_table(
    table_name='endpoints',
    s3_location=f's3://{BUCKET_NAME}/telemetry/endpoints/',
    columns=[
        {'Name': 'endpoint_id', 'Type': 'int'},
        {'Name': 'name', 'Type': 'string'},
        {'Name': 'region', 'Type': 'string'},
        {'Name': 'status', 'Type': 'string'},
        {'Name': 'category', 'Type': 'string'}
    ]
)

create_glue_table(
    table_name='alerts',
    s3_location=f's3://{BUCKET_NAME}/telemetry/alerts/',
    columns=[
        {'Name': 'alert_id', 'Type': 'int'},
        {'Name': 'endpoint_id', 'Type': 'int'},
        {'Name': 'severity', 'Type': 'string'},
        {'Name': 'message', 'Type': 'string'},
        {'Name': 'created_at', 'Type': 'string'}
    ]
)

print(f"Glue catalog: {GLUE_DB}.endpoints and {GLUE_DB}.alerts created")

Glue database created: citi_telemetry_db


Glue table created: citi_telemetry_db.endpoints
Glue table created: citi_telemetry_db.alerts
Glue catalog: citi_telemetry_db.endpoints and citi_telemetry_db.alerts created


## Athena — Serverless SQL on S3

Athena is a Presto/Trino-style SQL engine that reads files directly from S3 through the Glue catalog. You do not provision a database server. You pay for bytes scanned, the results are written back to S3, and the operational model is extremely light for ad-hoc SQL and exploratory analytics.

In [5]:
athena = boto3.client('athena')

def run_athena(sql: str, database: str, output_bucket: str):
    try:
        response = athena.start_query_execution(
            QueryString=sql,
            QueryExecutionContext={'Database': database},
            ResultConfiguration={'OutputLocation': f's3://{output_bucket}/results/'}
        )
        query_execution_id = response['QueryExecutionId']
    except ClientError as e:
        print(f'Error starting Athena query: {e}')
        return []

    state = 'QUEUED'
    for _ in range(15):
        try:
            status_response = athena.get_query_execution(QueryExecutionId=query_execution_id)
            state = status_response['QueryExecution']['Status']['State']
            if state in ('SUCCEEDED', 'FAILED', 'CANCELLED'):
                break
            time.sleep(2)
        except ClientError as e:
            print(f'Error polling Athena query {query_execution_id}: {e}')
            return []

    if state != 'SUCCEEDED':
        print(f'Athena query did not succeed. Final state: {state}')
        return []

    try:
        results = athena.get_query_results(QueryExecutionId=query_execution_id)
    except ClientError as e:
        print(f'Error fetching Athena results: {e}')
        return []

    rows = results['ResultSet']['Rows']
    if not rows:
        return []

    headers = [col.get('VarCharValue', '') for col in rows[0]['Data']]
    parsed = []
    for row in rows[1:]:
        values = [item.get('VarCharValue', '') for item in row['Data']]
        if len(values) < len(headers):
            values += [''] * (len(headers) - len(values))
        parsed.append(dict(zip(headers, values)))
    return parsed

q1 = 'SELECT severity, COUNT(*) as cnt FROM alerts GROUP BY severity ORDER BY cnt DESC'
q2 = 'SELECT region, COUNT(DISTINCT endpoint_id) as endpoints FROM endpoints GROUP BY region ORDER BY endpoints DESC'
q3 = '''
SELECT e.region, a.severity, COUNT(*) as alert_count
FROM alerts a
JOIN endpoints e ON a.endpoint_id = e.endpoint_id
GROUP BY e.region, a.severity
ORDER BY alert_count DESC
LIMIT 20
'''

for i, sql in enumerate([q1, q2, q3], start=1):
    print(f'\n--- Query {i} ---')
    print(sql)
    result_rows = run_athena(sql, GLUE_DB, ATHENA_BUCKET)
    if result_rows:
        print(pd.DataFrame(result_rows))
    else:
        print('No rows returned')


--- Query 1 ---
SELECT severity, COUNT(*) as cnt FROM alerts GROUP BY severity ORDER BY cnt DESC


   severity   cnt
0      HIGH  6313
1       LOW  6254
2    MEDIUM  6248
3  CRITICAL  6185

--- Query 2 ---
SELECT region, COUNT(DISTINCT endpoint_id) as endpoints FROM endpoints GROUP BY region ORDER BY endpoints DESC


  region endpoints
0   SNG1      2579
1   NYC1      2489
2   LON1      2485
3   NYC2      2447

--- Query 3 ---

SELECT e.region, a.severity, COUNT(*) as alert_count
FROM alerts a
JOIN endpoints e ON a.endpoint_id = e.endpoint_id
GROUP BY e.region, a.severity
ORDER BY alert_count DESC
LIMIT 20



   region  severity alert_count
0    SNG1    MEDIUM        1656
1    SNG1       LOW        1640
2    SNG1  CRITICAL        1635
3    NYC1      HIGH        1589
4    SNG1      HIGH        1588
5    NYC1    MEDIUM        1580
6    NYC1       LOW        1580
7    NYC2      HIGH        1579
8    NYC2    MEDIUM        1559
9    LON1      HIGH        1557
10   NYC1  CRITICAL        1537
11   NYC2       LOW        1532
12   LON1  CRITICAL        1509
13   NYC2  CRITICAL        1504
14   LON1       LOW        1502
15   LON1    MEDIUM        1453


## Cost — What This Costs

| Service | How you're charged | This notebook |
|---------|-------------------|---------------|
| S3 | $0.023/GB/month + PUT requests | ~$0.01 for 25K rows in Parquet |
| Glue Catalog | First 1M objects free | Free |
| Athena | $5/TB scanned | ~$0.01 for 3 queries on 2 small files |
| Total this session | — | < $0.05 |

Parquet compression = 10x fewer bytes scanned vs CSV → 10x less cost

Delete all AWS resources created in this notebook to avoid ongoing charges

In [6]:
def delete_all_objects_in_bucket(bucket_name: str):
    try:
        paginator = s3.get_paginator('list_object_versions')
        found_any = False
        for page in paginator.paginate(Bucket=bucket_name):
            objects_to_delete = []
            for item in page.get('Versions', []):
                objects_to_delete.append({'Key': item['Key'], 'VersionId': item['VersionId']})
            for item in page.get('DeleteMarkers', []):
                objects_to_delete.append({'Key': item['Key'], 'VersionId': item['VersionId']})
            if objects_to_delete:
                found_any = True
                s3.delete_objects(Bucket=bucket_name, Delete={'Objects': objects_to_delete})
        if not found_any:
            simple = s3.list_objects_v2(Bucket=bucket_name)
            if 'Contents' in simple:
                s3.delete_objects(
                    Bucket=bucket_name,
                    Delete={'Objects': [{'Key': obj['Key']} for obj in simple['Contents']]}
                )
        print(f'All objects deleted from: {bucket_name}')
    except ClientError as e:
        print(f'Error deleting objects from {bucket_name}: {e}')

for bucket_name in [BUCKET_NAME, ATHENA_BUCKET]:
    delete_all_objects_in_bucket(bucket_name)
    try:
        s3.delete_bucket(Bucket=bucket_name)
        print(f'Bucket deleted: {bucket_name}')
    except ClientError as e:
        print(f'Error deleting bucket {bucket_name}: {e}')

for table_name in ['endpoints', 'alerts']:
    try:
        glue.delete_table(DatabaseName=GLUE_DB, Name=table_name)
        print(f'Glue table deleted: {GLUE_DB}.{table_name}')
    except glue.exceptions.EntityNotFoundException:
        print(f'Glue table already absent: {GLUE_DB}.{table_name}')
    except ClientError as e:
        print(f'Error deleting Glue table {GLUE_DB}.{table_name}: {e}')

try:
    glue.delete_database(Name=GLUE_DB)
    print(f'Glue database deleted: {GLUE_DB}')
except glue.exceptions.EntityNotFoundException:
    print(f'Glue database already absent: {GLUE_DB}')
except ClientError as e:
    print(f'Error deleting Glue database {GLUE_DB}: {e}')

try:
    if pg_conn is not None:
        pg_conn.close()
        print('Postgres connection closed')
except Exception as e:
    print(f'Error closing Postgres connection: {e}')

print('Clean up complete — all AWS resources deleted')

All objects deleted from: citi-telemetry-lake-357811130281


Bucket deleted: citi-telemetry-lake-357811130281


All objects deleted from: citi-telemetry-athena-357811130281


Bucket deleted: citi-telemetry-athena-357811130281


Glue table deleted: citi_telemetry_db.endpoints


Glue table deleted: citi_telemetry_db.alerts


Glue database deleted: citi_telemetry_db
Postgres connection closed
Clean up complete — all AWS resources deleted


## What Just Happened

- exported Postgres → Parquet
- uploaded to S3
- Glue catalog created
- Athena SQL ran
- cost < $0.05
- cleaned up

This is the Citi AWS DE pattern: nightly Airflow DAG exports telemetry to S3, Glue crawler updates the catalog, Athena serves the risk team SQL access — no data warehouse needed.

Next: Run aws_de_concepts.md for Glue/EMR/Kinesis vocabulary.